In [45]:
# Data Ingestion

In [1]:
import os
from pathlib import Path

def has_project_files(path: Path) -> bool:
    try:
        return (path / "config" / "config.yaml").is_file() and (path / "src").is_dir()
    except OSError:
        return False

def find_project_root(start: Path = Path.cwd()) -> Path:
    try:
        start = start.resolve()
    except OSError:
        start = Path.cwd()

    seeds = [start, Path.home() / "Documents" / "projects" / "Summarizer"]
    candidates = []
    for seed in seeds:
        try:
            seed = seed.resolve()
        except OSError:
            pass
        for candidate in [seed, *seed.parents]:
            if candidate not in candidates:
                candidates.append(candidate)
    for candidate in candidates:
        if has_project_files(candidate):
            return candidate

    for candidate in candidates:
        try:
            children = list(candidate.iterdir())
        except OSError:
            continue
        for child in children:
            try:
                if child.is_dir() and has_project_files(child):
                    return child
            except OSError:
                continue
    raise FileNotFoundError("Could not find project root containing config/config.yaml")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
PROJECT_ROOT


PosixPath('/Users/tyrion/Documents/projects/Summarizer')

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

    


In [3]:
from src.Text_Summarizer.utils.common import read_yaml, create_directories
from src.Text_Summarizer.constants import *

In [4]:
class ConfigurationManager:
    def __init__(self, config_filepath: Path = CONFIG_FILE_PATH, params_filepath: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [5]:
import os
import urllib.request as request
import zipfile
from src.Text_Summarizer.logger import logger


In [6]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=str(self.config.source_URL),
                filename=str(self.config.local_data_file)
            )
            logger.info(f"{filename} downloaded! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {os.path.getsize(self.config.local_data_file) / (1024 * 1024)} MB")

    def unzip_and_clean(self):
        unzip_dir = self.config.unzip_dir
        os.makedirs(unzip_dir, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_dir)

In [7]:
config = ConfigurationManager()
data_ingestion_config= config.get_data_ingestion_config()
data_ingestion = DataIngestion(config=data_ingestion_config)

data_ingestion.download_data()
data_ingestion.unzip_and_clean()



[2026-07-05 15:41:46,720: INFO: common]: YAML file: /Users/tyrion/Documents/projects/Summarizer/config/config.yaml loaded successfully.
[2026-07-05 15:41:46,723: INFO: common]: YAML file: /Users/tyrion/Documents/projects/Summarizer/params.yaml loaded successfully.
[2026-07-05 15:41:46,724: INFO: common]: Directory created at: artifacts
[2026-07-05 15:41:46,725: INFO: common]: Directory created at: artifacts/data_ingestion
[2026-07-05 15:41:46,725: INFO: 2927394386]: File already exists of size: 7.537454605102539 MB
